# 实验三：手搓最小 LLM（CPU）

本 Notebook 对应实验指南全部必做内容，并完成进阶选做 **禁止重复 3-gram**。训练代码位于 `lab03_min_llm/min_llm.py`，所有结果隔离保存到 `lab03_outputs/`。

In [ ]:
import csv
import json
import math
import platform
import sys
from pathlib import Path

from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'lab03_min_llm':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'lab03_min_llm'))
import min_llm

print('操作系统:', platform.platform())
print('Python:', platform.python_version())
print('PyTorch:', min_llm.torch.__version__)
print('CUDA 可用:', min_llm.torch.cuda.is_available())
print('设备: CPU')
print('CPU 线程数:', min_llm.torch.get_num_threads())

## 1. 语料、字符级分词与标签错位

使用指南内置 72 首诗，字符级词表；训练样本 `x=data[i:i+T]`，标签 `y=data[i+1:i+T+1]`，逐位置预测下一个字符。

In [ ]:
text = min_llm.build_corpus()
tokenizer = min_llm.CharTokenizer(text)
data = min_llm.torch.tensor(tokenizer.encode(text), dtype=min_llm.torch.long)
generator = min_llm.torch.Generator().manual_seed(42)
batch_x, batch_y = min_llm.get_batch(data, 128, 2, generator)
print('语料字符数:', len(text))
print('词表大小 V:', tokenizer.vocab_size)
print('x/y 形状:', tuple(batch_x.shape), tuple(batch_y.shape))
print('标签错开一位验证:', bool(min_llm.torch.equal(batch_x[:, 1:], batch_y[:, :-1])))

## 2. 基线架构与参数量手算

基线为 `V=699, C=128, H=4, L=2, T=128`，Pre-Norm Transformer，Q/K/V 合并投影，带因果掩码，词嵌入与输出层权重共享。近似公式：

`V×C + T×C + L×12C² = 699×128 + 128×128 + 2×12×128² = 499,072`。

In [ ]:
baseline_model = min_llm.build_model(min_llm.BASELINE, tokenizer.vocab_size)
exact_parameters = sum(parameter.numel() for parameter in baseline_model.parameters())
estimated_parameters = min_llm.parameter_estimate(min_llm.BASELINE, tokenizer.vocab_size)
relative_error = abs(exact_parameters - estimated_parameters) / exact_parameters * 100
with min_llm.torch.no_grad():
    _, initial_loss = baseline_model(batch_x, batch_y)
print('公式估算参数量:', f'{estimated_parameters:,}')
print('代码精确参数量:', f'{exact_parameters:,}')
print('相对误差:', f'{relative_error:.3f}%')
print('初始 loss:', f'{initial_loss.item():.4f}')
print('ln(V):', f'{math.log(tokenizer.vocab_size):.4f}')

## 3. 必做控制变量矩阵

基线训练 2000 步；exp1-exp5 每个配置训练 1000 步。exp1、exp2、exp3 分别包含指南指定的两个实验值，因此合计 1 次基线和 8 次对照训练。exp6 固定基线权重，仅改变采样参数。

In [ ]:
matrix = ['|组别|配置键|相对基线改动|训练步数|', '|---|---|---|---:|']
for config in min_llm.EXPERIMENTS:
    matrix.append(f'|{config.group}|{config.key}|{config.change}|{config.iterations}|')
display(Markdown('\n'.join(matrix)))

## 4. 运行全部必做训练和进阶选做

以下调用会断点式跳过已经完整保存的实验，避免意外覆盖。若确需全部重跑，才在命令行显式使用 `--force`。

In [ ]:
results = min_llm.run_full_suite(force=False)

## 5. 对照实验结果表

最终 loss 同时列出最后一个 batch 和最后 100 步均值；报告表按指南可填写 `final_loss`，分析时参考更稳定的 `last100_mean_loss`。

In [ ]:
with (ROOT / 'lab03_outputs' / 'results.csv').open(encoding='utf-8-sig') as file:
    rows = list(csv.DictReader(file))
quality_stars = {
    'baseline': 5, 'exp1_lr1e2': 4, 'exp1_lr1e4': 2,
    'exp2_layer1': 5, 'exp2_layer4': 4,
    'exp3_embd64': 3, 'exp3_embd256': 5,
    'exp4_block32': 4, 'exp5_no_pos': 3,
}
table = ['|组别|配置|改动|参数量|最终loss|末100步均值|耗时(min)|生成质量|', '|---|---|---|---:|---:|---:|---:|---:|']
for row in rows:
    table.append(
        f"|{row['group']}|{row['key']}|{row['change']}|{int(row['parameters']):,}|"
        f"{float(row['final_loss']):.4f}|{float(row['last100_mean_loss']):.4f}|"
        f"{float(row['training_seconds'])/60:.2f}|{'★' * quality_stars[row['key']]}|"
    )
display(Markdown('\n'.join(table)))

## 6. 温度与 top-k 采样（必做 exp6）

固定同一基线模型，对比温度 `0.5/1.0/1.5` 及 top-k `5/20/不限`。每个配置生成 3 段；`unique_char_ratio` 辅助观察多样性，`repeated_trigram_rate` 辅助观察重复。生成原文保存在 `generated_sampling.txt`。

In [ ]:
with (ROOT / 'lab03_outputs' / 'sampling_results.csv').open(encoding='utf-8-sig') as file:
    sampling_rows = list(csv.DictReader(file))
sampling_table = ['|配置|temperature|top-k|平均字符多样性|平均重复3-gram率|', '|---|---:|---:|---:|---:|']
for name in dict.fromkeys(row['config'] for row in sampling_rows):
    group = [row for row in sampling_rows if row['config'] == name]
    diversity = sum(float(row['unique_char_ratio']) for row in group) / len(group)
    repetition = sum(float(row['repeated_trigram_rate']) for row in group) / len(group)
    sampling_table.append(f"|{name}|{group[0]['temperature']}|{group[0]['top_k']}|{diversity:.3f}|{repetition:.3f}|")
display(Markdown('\n'.join(sampling_table)))

## 7. 进阶选做：禁止重复 3-gram

生成时若当前末尾两个字符曾作为某个 3-gram 的前缀出现，则屏蔽所有曾跟随该前缀的字符。为让重复现象可观察，固定基线模型并采用低熵设置 `temperature=0.5, top_k=1, 生成400字`；提示词和随机种子保持一致，唯一变量是重复约束。

In [ ]:
with (ROOT / 'lab03_outputs' / 'repetition_results.csv').open(encoding='utf-8-sig') as file:
    repetition_rows = list(csv.DictReader(file))
optional_table = ['|方法|平均字符多样性|平均重复3-gram率|', '|---|---:|---:|']
for method in dict.fromkeys(row['method'] for row in repetition_rows):
    group = [row for row in repetition_rows if row['method'] == method]
    diversity = sum(float(row['unique_char_ratio']) for row in group) / len(group)
    repetition = sum(float(row['repeated_trigram_rate']) for row in group) / len(group)
    optional_table.append(f'|{method}|{diversity:.3f}|{repetition:.3f}|')
display(Markdown('\n'.join(optional_table)))

## 8. 报告填写索引

- 环境表：第 1 个代码单元输出。
- 语料、错位标签：第 2 节。
- 基线参数量手算：第 2 节及 `parameter_check.json`。
- 基线与 exp1-exp5：第 5 节、`results.csv`、对应 `loss_curve_*.png` 和 `generated_*.txt`。
- exp6 采样：第 6 节、`sampling_results.csv` 和 `generated_sampling.txt`。
- 进阶任务：第 7 节、`repetition_results.csv` 和 `generated_repetition_compare.txt`。

报告中的生成质量星级和文字结论应以保存的真实生成原文为依据，不凭 loss 单独判断。

## 9. 基于本次真实结果的归因要点

- 基线初始 loss 为 6.5607，与 `ln(699)=6.5497` 接近；最终 loss 为 0.0322，末 100 步均值 0.0246，说明小语料被充分拟合。生成文本格式连贯，但大段复现训练诗句，体现明显背诵。
- `lr=1e-2` 收敛较快但曲线前期波动更大，末 100 步均值 0.0495；`lr=1e-4` 在 1000 步后仍为 0.7185，主要问题是收敛过慢。
- 1 层与 4 层在 1000 步的末 100 步均值分别为 0.0296 和 0.0298，表明在极小语料上增加深度没有带来明显训练 loss 优势，却显著增加参数量和耗时。
- 64 维模型容量较小，末 100 步均值 0.0564；256 维降至 0.0264，但参数量增至 1,791,744。
- 上下文缩短至 32 后末 100 步均值升至 0.1044，长距离格式和跨句衔接更弱。去掉位置编码后均值为 0.0489，仍能学习字符共现，却更容易拼接错序。
- 低温或较小 top-k 输出更确定、通顺，但更接近逐段背诵；高温 1.5 出现错字、重复和语义跳跃。综合可读性与少量变化，本次可将 `temperature=1.0, top_k=20` 作为折中设置。
- 进阶任务中，原始低熵生成的平均重复 3-gram 率为 0.00586，禁止重复 3-gram 后为 0；约束能消除局部循环，但不会解决小语料导致的整体背诵。

## 10. 可如实记录的 CODEX 协作

1. 按指南拆分基线、8 个训练配置、采样矩阵和进阶任务，并核对单一变量。
2. 实现并检查 Q/K/V 多头拆分、因果掩码、Pre-Norm 残差和嵌入/输出权重共享。
3. 定位 Jupyter 默认内核缺少 PyTorch 的问题，切换到已有 `myenv`，未安装新环境。
4. 根据真实曲线和生成文本核对 loss、生成星级及温度/top-k 现象，没有仅凭预期填写结论。
5. 发现原选做样本重复不明显后，在不重训的前提下改用固定低熵采样，使 3-gram 约束具备可比较的前后证据。

In [ ]:
print('===== 基线真实生成原文：春/月各两段 =====')
print((ROOT / 'lab03_outputs' / 'generated_base.txt').read_text(encoding='utf-8'))
print('\n===== 各采样配置首段节选 =====')
for name in dict.fromkeys(row['config'] for row in sampling_rows):
    row = next(item for item in sampling_rows if item['config'] == name)
    print(f"[{name}] {row['text'][:160]}")